In [29]:
path = "../data/MALASPINA_LEG1/D20110113-T074949.raw"

In [32]:
import numpy as np
import xarray as xr
from scipy.ndimage import uniform_filter1d
import echopype as ep


def detect_bottom_from_Sv(
    Sv_da,
    min_depth=100,     # ignore near-surface scattering
    max_depth=6000,    # deepest expected bottom
    min_Sv=-65,        # threshold for likely seabed return
    smooth_window=15,  # vertical smoothing (samples)
    gradient_window=8  # for derivative bottom pick
):
    """
    Robust bottom detection for EK60 38 kHz Sv echograms.
    
    Parameters
    ----------
    Sv_da : xarray.DataArray
        echopype Sv DataArray with dims (ping_time, range)
    min_depth : float (m)
    max_depth : float (m)
    min_Sv : float
        threshold for seabed scattering (~ -60 to -70 dB)
    smooth_window : int
        smoothing kernel along range
    gradient_window : int
        window for gradient-based bottom picking
        
    Returns
    -------
    bottom_depth : np.ndarray
        array of bottom depth per ping (meters)
    """

    Sv = Sv_da.values   # shape: (pings, ranges)
    ranges = Sv_da["range"].values  # 1D array (meters)

    # restrict to valid depth interval
    mask = (ranges >= min_depth) & (ranges <= max_depth)
    Sv = Sv[:, mask]
    r = ranges[mask]

    # 1. smooth in range direction (deep conditions need smoothing)
    Sv_smooth = uniform_filter1d(Sv, size=smooth_window, axis=1)

    # 2. apply threshold to isolate strong seabed region
    seabed_mask = Sv_smooth > min_Sv

    # allocate output
    bottom_depth = np.full(Sv.shape[0], np.nan)

    for i in range(Sv.shape[0]):
        col = Sv_smooth[i, :]
        mask_i = seabed_mask[i, :]

        if not np.any(mask_i):
            continue  # no detectable bottom for this ping

        # 3. candidate region = connected region above threshold
        idx = np.where(mask_i)[0]

        # 4. choose the deepest candidate region
        deepest_idx = idx[-1]

        # 5. refine bottom using local gradient peak
        i0 = max(0, deepest_idx - gradient_window)
        i1 = min(len(col), deepest_idx + gradient_window)
        window = col[i0:i1]

        # gradient -> seabed strong rise
        grad = np.gradient(window)
        local_peak = np.argmax(grad)
        refined = i0 + local_peak

        bottom_depth[i] = r[refined]

    return bottom_depth



In [41]:
FileName = 'D20110113-T074949'
ed = ep.open_raw("../data/MALASPINA_LEG1/" + FileName + '.raw', sonar_model="EK60")  # for EK60 file

Sv = ep.calibrate.compute_Sv(ed)
#Sv_da = Sv["Sv"].sel(channel=0)  # use 38 kHz channel
Sv_da = Sv["Sv"].isel(channel=0)
#bottom_depth = detect_bottom_from_Sv(Sv_da)


sample_interval = float(ed["Sonar/Beam_group1"]["sample_interval"].values[0][0])
sound_speed = float(ed["Environment"]["sound_speed_indicative"].values[0][0])

depth_m = np.arange(Sv_da.shape[1]) * (sample_interval * sound_speed / 2)

In [42]:
depth_m

array([0.00000000e+00, 1.96746187e-01, 3.93492374e-01, ...,
       9.99470630e+02, 9.99667376e+02, 9.99864122e+02], shape=(5083,))

In [43]:

print("sample_interval shape:", ed["Sonar/Beam_group1"]["sample_interval"].values.shape)
print("sound_speed_indicative shape:", ed["Environment"]["sound_speed_indicative"].values.shape)


sample_interval shape: (1, 508)
sound_speed_indicative shape: (1, 508)
